# E12: Model Lock (model_registry.json)

**Spec sections**: 7.7 (model lock), 8.1 (pre-registered decision rules), F.3 (registry / UC), F.4 (OOT grant, recommended).  
**Inputs**: E11 model files + `models/sha256.json`, `models/model_spec.json`, `params/chosen_params.json`, feature lists, L1 binning objects, `oof_predictions.parquet`, run summaries of E4/E10/E11, `config/signoff.yaml`.  
**Outputs**: `model_registry.json` (artefacts root), `registry_history/` (previous locked versions), `reports/run_summaries/s04_lock.json`, MLflow run.

1. Build the registry: version, hashes of every model / binning / feature-list / params / OOF file, features, params, final rounds, DEV OOF and T-fold metrics, EBM shape check, R1–R4 and selection logic with thresholds copied from `config.selection`, bootstrap settings, Platt calibration (fit on DEV OOF), `config.strategy_overlay`.
2. Immutability: a locked registry is never changed. Any difference needs a new `config.registry.version`; the old version is archived.
3. Lock only if `decision_rules_signed_by` and `signed_date` are filled and every hash re-verifies; otherwise write `locked: false` and stop.
4. Optional UC registration (`config.registry.register_in_uc`).

In [0]:
%pip install optbinning "numpy<2" "pyarrow==14.0.1" "statsmodels>=0.14.4" lightgbm interpret-core "scipy>=1.12,<1.14" --quiet
dbutils.library.restartPython()

In [0]:
import sys, os, json, time, shutil, tempfile, hashlib, importlib, copy, warnings
from datetime import datetime, date
sys.path.insert(0, os.getcwd())          # notebook lives in src/
import common
importlib.reload(common)
PROJECT_ROOT = common.find_project_root(os.getcwd())

import numpy as np
import pandas as pd
import yaml

cfg = common.load_config(os.path.join(PROJECT_ROOT, "config", "config.yaml"))
SEED = cfg["random_seed"]
np.random.seed(SEED)

ARTIFACTS  = common.artifacts_path(cfg)
REPORTS    = os.path.join(ARTIFACTS, "reports")
RUN_SUMS   = os.path.join(REPORTS, "run_summaries")
FEAT_LISTS = os.path.join(ARTIFACTS, "feature_lists")
BIN_DIR    = os.path.join(ARTIFACTS, "binning", "L1")
PARAMS_DIR = os.path.join(ARTIFACTS, "params")
MODELS_DIR = os.path.join(ARTIFACTS, "models")
FOLDS_DIR  = os.path.join(ARTIFACTS, "folds")
OOF_PATH   = os.path.join(ARTIFACTS, "oof_predictions.parquet")
REGISTRY_PATH = os.path.join(ARTIFACTS, "model_registry.json")
HISTORY_DIR   = os.path.join(ARTIFACTS, "registry_history")
SIGNOFF_PATH  = os.path.join(PROJECT_ROOT, cfg["review_files"]["signoff"])
os.makedirs(RUN_SUMS, exist_ok=True)

cat, sch = cfg["databricks"]["catalog"], cfg["databricks"]["schema"]
MASTER_TBL = f"{cat}.{sch}.acard_master"
FLOG_TBL   = f"{cat}.{sch}.acard_feature_log"
id_col = cfg["columns"]["id"]

REG_CFG = cfg["registry"]
VERSION = str(REG_CFG["version"])
SEL = cfg["selection"]
CAL = cfg["calibration"]
OVERLAY = cfg["strategy_overlay"]

GIT_COMMIT = common.git_commit(PROJECT_ROOT)
CONFIG_HASH = common.config_hash(cfg)
FUNNEL_HASH = hashlib.sha256(json.dumps(cfg["funnel"], sort_keys=True).encode()).hexdigest()

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def read_json(p):
    with open(p) as fh:
        return json.load(fh)

def write_json(obj, dst):
    """Write to a local temp file, then copy to the UC volume (FUSE does not allow chmod/utime)."""
    tmp = os.path.join(tempfile.mkdtemp(prefix="e12_"), os.path.basename(dst))
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copyfile(tmp, dst)

def canonical(obj):
    return json.loads(json.dumps(obj, sort_keys=True, default=lambda o: o.item() if hasattr(o, "item") else str(o)))

_t0 = time.time()
print(f"Project root : {PROJECT_ROOT}")
print(f"Git commit   : {GIT_COMMIT[:12]} | Config hash: {CONFIG_HASH[:12]} | Funnel hash: {FUNNEL_HASH[:12]}")
print(f"Registry     : {REGISTRY_PATH} | version {VERSION} | register_in_uc={REG_CFG['register_in_uc']}")
print(f"Selection    : {SEL}")
print(f"Calibration  : {CAL}")
print(f"Overlay      : {OVERLAY}")

In [0]:
# ── Upstream outputs (no OOT data is touched here) ──
S_L0    = read_json(os.path.join(RUN_SUMS, "s03_L0.json"))
S_TUNE  = read_json(os.path.join(RUN_SUMS, "s04_tune.json")) if os.path.exists(os.path.join(RUN_SUMS, "s04_tune.json")) else {}
S_TRAIN = read_json(os.path.join(RUN_SUMS, "s04_train.json"))
CP      = read_json(os.path.join(PARAMS_DIR, "chosen_params.json"))
SPEC    = read_json(os.path.join(MODELS_DIR, "model_spec.json"))
E11_SHA = read_json(os.path.join(MODELS_DIR, "sha256.json"))
A_FEATS = [r["feature"] for r in read_json(os.path.join(FEAT_LISTS, "model_a.json"))["features"]]
POOL    = [r["feature"] for r in read_json(os.path.join(FEAT_LISTS, "l4_pool.json"))]

problems = []
if S_TRAIN.get("pass_fail") != "PASS":
    problems.append(f"E11 run summary pass_fail = {S_TRAIN.get('pass_fail')}")
if S_L0["funnel_config_hash"] != FUNNEL_HASH:
    problems.append(f"funnel config hash {FUNNEL_HASH[:12]} != frozen E4 hash {S_L0['funnel_config_hash'][:12]}")
for name, h in (("E10 chosen_params", CP["config_sha256"]), ("E11 model_spec", SPEC["config_sha256"]),
                ("E11 run summary", S_TRAIN["config_hash"])):
    if h != CONFIG_HASH:
        problems.append(f"config hash changed since {name}: {h[:12]} -> {CONFIG_HASH[:12]}")
if not (A_FEATS == CP["A"]["features"] == SPEC["A"]["features"]):
    problems.append("Model A feature list differs between model_a.json, chosen_params.json and model_spec.json")
if not (POOL == CP["B"]["input_features"] == CP["C"]["input_features"] == SPEC["B"]["input_features"] == SPEC["C"]["input_features"]):
    problems.append("L4 pool differs between l4_pool.json, chosen_params.json and model_spec.json")
if int(SPEC["C"]["num_boost_round"]) != int(CP["C"]["final_rounds"]) or int(S_TRAIN["C_final_rounds"]) != int(CP["C"]["final_rounds"]):
    problems.append("LightGBM final rounds differ between E10 and E11")
for k in ("bootstrap", "r1_gini_lift_lower", "c_over_b_gini_lift_lower", "r3_tfold_std_ratio", "r3_q2_ks_drop_margin"):
    if k not in SEL:
        problems.append(f"config.selection.{k} missing")
if CAL.get("method") != "platt" or CAL.get("fit_on") != "dev_oof":
    problems.append(f"config.calibration must be platt fitted on dev_oof (spec 9.1); got {CAL}")
assert not problems, "E12 stopped:\n  - " + "\n  - ".join(problems)

# Strategy-overlay external score vs the feature log (information for the reviewer; the overlay is not a model feature)
_ext = OVERLAY.get("external_score")
_fl = spark.table(FLOG_TBL).filter(f"feature = '{_ext}'").select("feature", "L0_pass", "dropped_at", "drop_rule").toPandas()
OVERLAY_NOTE = None
if len(_fl) and str(_fl["drop_rule"].iat[0]) == "COMPLIANCE":
    OVERLAY_NOTE = (f"strategy_overlay.external_score = {_ext} is dropped at {_fl['dropped_at'].iat[0]} for COMPLIANCE as a model "
                    f"feature; its use in the strategy overlay needs compliance confirmation.")
    print(f"\u26a0 WARNING: {OVERLAY_NOTE}")

print(f"E11 PASS; config/funnel hashes consistent; Model A {len(A_FEATS)} features; L4 pool {len(POOL)}; "
      f"C final rounds {CP['C']['final_rounds']}; E11 files hashed {len(E11_SHA)}")
print(f"EBM shape check (E11): {S_TRAIN['ebm_shape_check']} | reversals {S_TRAIN['ebm_reversals']}")

In [0]:
# ── Evidence that OOT cannot be read before the lock. A tiny synthetic frame is used: load_window checks the
#    registry before touching any data, so no OOT record is read here. ──
_probe = pd.DataFrame({"window": ["DEV", "OOT"], id_col: ["x", "y"]})
EXISTING = read_json(REGISTRY_PATH) if os.path.exists(REGISTRY_PATH) else None
PRE_LOCK_TEST = {"registry_existed": EXISTING is not None,
                 "registry_locked": bool(EXISTING and EXISTING.get("locked"))}

def _raises(reg_path):
    try:
        common.load_window(_probe, "OOT", cfg, registry_path=reg_path)
        return False, None
    except PermissionError as e:
        return True, str(e)

if not PRE_LOCK_TEST["registry_locked"]:
    PRE_LOCK_TEST["raised"], PRE_LOCK_TEST["message"] = _raises(REGISTRY_PATH)
    PRE_LOCK_TEST["basis"] = "actual registry path"
else:
    # Re-run after an earlier lock: test the guard on a temporary unlocked registry instead
    _tmp_reg = os.path.join(tempfile.mkdtemp(prefix="e12_"), "model_registry.json")
    with open(_tmp_reg, "w") as fh:
        json.dump({"locked": False}, fh)
    PRE_LOCK_TEST["raised"], PRE_LOCK_TEST["message"] = _raises(_tmp_reg)
    PRE_LOCK_TEST["basis"] = "temporary unlocked registry (a locked registry already exists)"
print(PRE_LOCK_TEST)
assert PRE_LOCK_TEST["raised"], "load_window(master, 'OOT') did NOT raise before the lock"

In [0]:
# ── Files under the lock: E11 model files (incl. model_spec.json), L1 binning of every pool feature, feature lists,
#    chosen params, folds and the OOF predictions used for Platt calibration. ──
FILES = {}
def _add(rel_to_artifacts):
    p = os.path.join(ARTIFACTS, rel_to_artifacts)
    assert os.path.exists(p), f"missing file under the lock: {p}"
    FILES[rel_to_artifacts] = {"path": p, "bytes": os.path.getsize(p), "sha256": sha256_file(p)}

for rel in sorted(E11_SHA):
    _add(f"models/{rel}")
for f in POOL:
    _add(f"binning/L1/{f}.pkl")
for rel in ("feature_lists/model_a.json", "feature_lists/l4_pool.json", "params/chosen_params.json",
            "folds/r_folds.parquet", "folds/t_folds.parquet", "oof_predictions.parquet"):
    _add(rel)

# E11 hashes must still match the files (independent of the hashes just computed)
E11_MISMATCH = [rel for rel, h in E11_SHA.items() if FILES[f"models/{rel}"]["sha256"] != h]
assert not E11_MISMATCH, f"Model files changed since E11 (sha256 differs): {E11_MISMATCH}"

r = lambda x: round(float(x), 6)
DECISION_RULES = {
    "R1_discrimination": {
        "requirement": "Lower bound of the 95% CI of the paired-bootstrap OOT delta Gini (X - A) > threshold",
        "threshold": SEL["r1_gini_lift_lower"], "source": "config.selection.r1_gini_lift_lower (TBC-09)"},
    "R2_calibration": {
        "requirement": "After Platt calibration (spec 9.1), X's OOT log loss <= A's OOT log loss (point estimate)",
        "threshold": None},
    "R3_time_stability": {
        "requirement": "X's T1-T3 AUC std <= ratio x A's; and X's KS drop 2025Q1 -> 2025Q2 <= A's drop + margin",
        "tfold_std_ratio": SEL["r3_tfold_std_ratio"], "q2_ks_drop_margin": SEL["r3_q2_ks_drop_margin"],
        "oot_report_splits": cfg["oot_report_splits"]},
    "R4_explainability": {
        "requirement": "Reason codes generated per Chapter 13 and approved by compliance; EBM shape check (spec 7.4) passed",
        "threshold": "pass/fail"},
}
SELECTION_LOGIC = [
    "A challenger X in {B, C} is better than the benchmark A only if it meets all of R1-R4.",
    "Neither B nor C qualifies -> choose A.",
    "Exactly one qualifies -> choose that model.",
    (f"Both qualify -> paired-bootstrap CI of delta Gini (C - B): lower bound > {SEL['c_over_b_gini_lift_lower']} "
     f"-> choose C, otherwise choose B."),
    "Score direction: comparisons use PD (higher = riskier). Thresholds MUST NOT change after OOT is read.",
]

CONTENT = {
    "project": cfg["project"],
    "version": VERSION,
    "git_commit": GIT_COMMIT,
    "config_sha256": CONFIG_HASH,
    "funnel_config_hash": FUNNEL_HASH,
    "code_versions": SPEC.get("versions", {}),
    "features": {"A": A_FEATS, "BC_pool": POOL, "BC_columns": SPEC["C"]["columns"]},
    "files": {k: {"path": v["path"], "bytes": v["bytes"], "sha256": v["sha256"]} for k, v in FILES.items()},
    "params": {
        "A": {"binning_params": CP["A"]["binning_params"], "special_codes": CP["A"]["special_codes"],
              "categorical_features": CP["A"]["categorical_features"], "estimator": CP["A"]["estimator"],
              "binning_fit_params": SPEC["A"]["binning_fit_params"], "woe_transform": SPEC["A"]["woe_transform"],
              "scaling": SPEC["A"]["scaling"], "factor": SPEC["A"]["factor"], "offset": SPEC["A"]["offset"]},
        "B": {"params": CP["B"]["params"], "e10_trial": CP["B"]["trial"], "monotonized": SPEC["B"]["monotonized"]},
        "C": {"params": CP["C"]["params"], "e10_trial": CP["C"]["trial"],
              "monotone_constraints": SPEC["C"]["monotone_constraints"]},
        "raw_preparation": SPEC.get("raw_preparation"),
    },
    "lightgbm_rounds": {"final_rounds": int(CP["C"]["final_rounds"]), "stage3_best_iter": int(CP["C"]["stage3_best_iter"]),
                        "round_multiplier": CP["C"]["round_multiplier"]},
    "dev_oof_metrics": {m: {"auc": r(S_TRAIN[f"{m}_oof_auc"]), "gini": r(S_TRAIN[f"{m}_oof_gini"]),
                            "ks": r(S_TRAIN[f"{m}_oof_ks"])} for m in "ABC"},
    "t_fold": {m: {"t_auc_mean": r(S_TRAIN[f"{m}_t_auc_mean"]), "t_auc_std": r(S_TRAIN[f"{m}_t_auc_std"])} for m in "ABC"},
    "ebm_shape_check": {"result": S_TRAIN["ebm_shape_check"], "reversals": S_TRAIN["ebm_reversals"]},
    "pre_registered": {
        "decision_rules": DECISION_RULES,
        "selection_logic": SELECTION_LOGIC,
        "selection_config": copy.deepcopy(SEL),
        "bootstrap": copy.deepcopy(SEL["bootstrap"]),
        "calibration": {"method": CAL["method"], "fit_on": CAL["fit_on"], "oof_file": "oof_predictions.parquet",
                        "formula": "logit(PD) = a + b * logit(p_raw)", "isotonic": "prohibited (spec 9.1)"},
        "strategy_overlay": copy.deepcopy(OVERLAY),
    },
    "source_runs": {"E4_L0": S_L0.get("run_time"), "E10_tune": CP.get("run_time"), "E11_train": S_TRAIN.get("run_time"),
                    "E11_mlflow_run_id": S_TRAIN.get("mlflow_run_id")},
    "notes": [n for n in [OVERLAY_NOTE] if n],
}
CONTENT = canonical(CONTENT)
print(f"Hashed {len(FILES)} files ({sum(v['bytes'] for v in FILES.values()):,} bytes); E11 hashes re-verified")
print(json.dumps(CONTENT["pre_registered"]["decision_rules"], indent=1))

In [0]:
# ── A locked registry never changes. Volatile fields (timestamps, run ids, UC registration) are not compared. ──
# "oot_evaluation" is appended by E13 after the lock and is not part of the locked content
VOLATILE = {"created_at", "locked_at", "lock_run", "uc_registration", "mlflow_run_id", "signoff", "locked",
            "oot_evaluation"}

def _diff(a, b, path=""):
    out = []
    if isinstance(a, dict) and isinstance(b, dict):
        for k in sorted(set(a) | set(b)):
            if k not in a or k not in b:
                out.append(f"{path}/{k}: {'added' if k not in a else 'removed'}")
            else:
                out += _diff(a[k], b[k], f"{path}/{k}")
    elif a != b:
        out.append(f"{path}: {str(a)[:80]} -> {str(b)[:80]}")
    return out

ACTION = "write"            # write | unchanged | new_version
ARCHIVED = None
if EXISTING and EXISTING.get("locked"):
    old_content = {k: v for k, v in EXISTING.items() if k not in VOLATILE}
    if str(EXISTING.get("version")) == VERSION:
        diffs = _diff(old_content, CONTENT)
        ov_old = EXISTING.get("pre_registered", {}).get("strategy_overlay")
        if ov_old != CONTENT["pre_registered"]["strategy_overlay"]:
            raise RuntimeError(f"REFUSED: config.strategy_overlay differs from the locked registry v{VERSION} "
                               f"({ov_old} -> {CONTENT['pre_registered']['strategy_overlay']}). "
                               f"Restore the config or set a new config.registry.version.")
        if diffs:
            raise RuntimeError(f"REFUSED: registry v{VERSION} is locked and {len(diffs)} field(s) would change:\n  "
                               + "\n  ".join(diffs[:20])
                               + "\nA change needs a new config.registry.version (spec 7.7) and disclosure in the model document.")
        ACTION = "unchanged"
    else:
        ARCHIVED = os.path.join(HISTORY_DIR, f"model_registry_v{EXISTING['version']}.json")
        if os.path.exists(os.path.join(HISTORY_DIR, f"model_registry_v{VERSION}.json")):
            raise RuntimeError(f"REFUSED: version {VERSION} already exists in {HISTORY_DIR}; choose a new version")
        ACTION = "new_version"
print(f"Existing registry: {'none' if not EXISTING else ('locked v' + str(EXISTING.get('version')) if EXISTING.get('locked') else 'unlocked v' + str(EXISTING.get('version')))} "
      f"-> action = {ACTION}" + (f" (old version archived to {ARCHIVED})" if ARCHIVED else ""))

In [0]:
with open(SIGNOFF_PATH) as fh:
    SIGNOFF = yaml.safe_load(fh) or {}

def _filled(v):
    return v is not None and str(v).strip() not in ("", "null", "None")

missing = []
if not _filled(SIGNOFF.get("decision_rules_signed_by")):
    missing.append("decision_rules_signed_by is empty in config/signoff.yaml")
signed_date = None
if not _filled(SIGNOFF.get("signed_date")):
    missing.append("signed_date is empty in config/signoff.yaml")
else:
    try:
        signed_date = pd.Timestamp(str(SIGNOFF["signed_date"])).date()
        if signed_date > date.today():
            missing.append(f"signed_date {signed_date} is in the future")
    except Exception:
        missing.append(f"signed_date '{SIGNOFF['signed_date']}' is not a valid date")

# Re-verify every hash from disk immediately before locking (second, independent read)
REVERIFY = {k: sha256_file(v["path"]) == v["sha256"] for k, v in CONTENT["files"].items()}
bad_hash = [k for k, ok in REVERIFY.items() if not ok]
if bad_hash:
    missing.append(f"{len(bad_hash)} file hash(es) do not re-verify: {bad_hash}")

now = datetime.now().isoformat(timespec="seconds")
if ACTION == "unchanged":
    REGISTRY = EXISTING
    print(f"Registry v{VERSION} already locked at {EXISTING.get('locked_at')} with identical content; not rewritten.")
    assert not bad_hash, f"Locked files changed on disk: {bad_hash}"
else:
    if ACTION == "new_version":
        os.makedirs(HISTORY_DIR, exist_ok=True)
        shutil.copyfile(REGISTRY_PATH, ARCHIVED)              # keep the previous locked version
    REGISTRY = {**CONTENT, "created_at": now, "locked": False,
                "lock_run": {"module": "E12_lock", "run_time": now, "git_commit": GIT_COMMIT}}
    write_json(REGISTRY, REGISTRY_PATH)                       # content first, unlocked
    if missing:
        print("Registry written with locked: false. Missing for the lock:\n  - " + "\n  - ".join(missing))
        raise RuntimeError("E12 stopped, model NOT locked: " + "; ".join(missing))
    REGISTRY["signoff"] = {"decision_rules_signed_by": str(SIGNOFF["decision_rules_signed_by"]),
                           "signed_date": str(signed_date), "model_owner": SIGNOFF.get("model_owner"),
                           "source": os.path.relpath(SIGNOFF_PATH, PROJECT_ROOT),
                           "source_sha256": sha256_file(SIGNOFF_PATH)}
    REGISTRY["locked_at"] = datetime.now().isoformat(timespec="seconds")
    REGISTRY["locked"] = True                                 # finally (spec 7.7)
    write_json(REGISTRY, REGISTRY_PATH)
    print(f"\u2705 Registry v{VERSION} LOCKED at {REGISTRY['locked_at']} (signed by {REGISTRY['signoff']['decision_rules_signed_by']} "
          f"on {REGISTRY['signoff']['signed_date']}); {len(REVERIFY)} file hashes re-verified")

In [0]:
# ── Spec F.3: optionally register the locked A/B/C in UC with alias "locked" (config.registry.register_in_uc) ──
UC_RESULT = {"enabled": bool(REG_CFG.get("register_in_uc")), "models": {}}
if UC_RESULT["enabled"] and REGISTRY.get("locked"):
    import mlflow
    from mlflow.tracking import MlflowClient
    from mlflow.models.signature import ModelSignature
    from mlflow.types.schema import Schema, ColSpec

    class LockedModel(mlflow.pyfunc.PythonModel):
        """Thin wrapper around the locked E11 files; returns the raw (uncalibrated) Bad probability."""
        def __init__(self, kind):
            self.kind = kind
        def load_context(self, context):
            import pickle, lightgbm as lgb, pandas as pd
            a = context.artifacts
            self.fdict = pd.read_csv(a["feature_dict"])
            if self.kind == "A":
                self.bp = pickle.load(open(a["binning_process"], "rb")); self.lr = pickle.load(open(a["lr"], "rb"))
            elif self.kind == "B":
                self.m = pickle.load(open(a["ebm"], "rb"))
            else:
                self.m = lgb.Booster(model_file=a["lgbm"])
        def predict(self, context, X):
            import numpy as np, common
            if self.kind == "A":
                w = np.asarray(self.bp.transform(X[A_FEATS], metric="woe", **SPEC["A"]["woe_transform"]), dtype=float)
                return self.lr.predict_proba(w)[:, 1]
            Xr = common.prepare_raw(X, POOL, self.fdict)[SPEC["C"]["columns"]]
            return self.m.predict_proba(Xr)[:, 1] if self.kind == "B" else self.m.predict(Xr)

    mlflow.set_registry_uri("databricks-uc")
    client = MlflowClient()
    fd = os.path.join(PROJECT_ROOT, "config", "feature_dict.csv")
    arts = {"A": {"binning_process": FILES["models/A/binning_process.pkl"]["path"], "lr": FILES["models/A/lr_sklearn.pkl"]["path"]},
            "B": {"ebm": FILES["models/B/ebm.pkl"]["path"]},
            "C": {"lgbm": FILES["models/C/lgbm.txt"]["path"]}}
    mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
    for m in "ABC":
        feats = A_FEATS if m == "A" else POOL
        sig = ModelSignature(inputs=Schema([ColSpec("double", f) for f in feats]), outputs=Schema([ColSpec("double")]))
        uc_name = f"{cat}.{sch}.acard_model_{m.lower()}"
        with mlflow.start_run(run_name=f"E12_register_{m}"):
            info = mlflow.pyfunc.log_model("model", python_model=LockedModel(m), artifacts={**arts[m], "feature_dict": fd},
                                           code_paths=[os.path.join(PROJECT_ROOT, "src", "common.py")],
                                           signature=sig, registered_model_name=uc_name)
        ver = info.registered_model_version
        client.set_registered_model_alias(uc_name, "locked", ver)
        UC_RESULT["models"][m] = {"name": uc_name, "version": str(ver), "alias": "locked"}
    REGISTRY["uc_registration"] = UC_RESULT
    write_json(REGISTRY, REGISTRY_PATH)
    print(f"Registered in UC: {UC_RESULT['models']}")
else:
    reason = "config.registry.register_in_uc = false" if not UC_RESULT["enabled"] else "registry not locked"
    print(f"UC registration skipped ({reason}). model_registry.json remains the record of the lock (spec F.3).")
print("Spec F.4 (recommended): grant SELECT on the OOT labels table to the development identity only now that the "
      "registry is locked; this is an admin action and is not performed by this notebook.")

In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(cfg["databricks"]["mlflow_experiment"])
with mlflow.start_run(run_name="E12_lock") as run:
    mlflow.set_tags({"module": "E12_lock", "git_commit": GIT_COMMIT, "registry_version": VERSION,
                     "locked": str(REGISTRY.get("locked")), "action": ACTION})
    mlflow.log_params({"config_hash": CONFIG_HASH[:12], "funnel_config_hash": FUNNEL_HASH[:12],
                       "n_files_locked": len(CONTENT["files"]), "r1_gini_lift_lower": SEL["r1_gini_lift_lower"],
                       "c_over_b_gini_lift_lower": SEL["c_over_b_gini_lift_lower"],
                       "r3_tfold_std_ratio": SEL["r3_tfold_std_ratio"], "r3_q2_ks_drop_margin": SEL["r3_q2_ks_drop_margin"],
                       "bootstrap_n": SEL["bootstrap"]["n"], "calibration": CAL["method"]})
    mlflow.log_artifact(REGISTRY_PATH)
    MLFLOW_RUN_ID = run.info.run_id
print(f"MLflow run {MLFLOW_RUN_ID}")

In [0]:
# ── Acceptance (E12): re-read the registry from the volume and check every criterion ──
print("=" * 60); print("ACCEPTANCE CRITERIA (E12 — model lock)"); print("=" * 60)
REG = read_json(REGISTRY_PATH)
sig = REG.get("signoff", {})
sign_ok = _filled(sig.get("decision_rules_signed_by")) and _filled(sig.get("signed_date"))
hash_now = {k: sha256_file(v["path"]) == v["sha256"] for k, v in REG["files"].items()}
sel_reg = REG["pre_registered"]["selection_config"]
rules = REG["pre_registered"]["decision_rules"]

# OOT read after the lock: master restricted to id + window (no labels are loaded by this check)
master_keys = spark.table(MASTER_TBL).select(id_col, "window").toPandas()
try:
    oot_after = common.load_window(master_keys, "OOT", cfg)
    POST_LOCK_OK, POST_LOCK_ROWS, POST_LOCK_ERR = True, len(oot_after), None
except PermissionError as e:
    POST_LOCK_OK, POST_LOCK_ROWS, POST_LOCK_ERR = False, 0, str(e)

checks = [
    (f"All {len(hash_now)} file hashes re-verify from disk", len(hash_now) > 0 and all(hash_now.values())),
    ("E11 sha256.json hashes match the registry",
     all(REG["files"][f"models/{k}"]["sha256"] == v for k, v in E11_SHA.items())),
    (f"locked = {REG.get('locked')} only with sign-off ({sig.get('decision_rules_signed_by')}, {sig.get('signed_date')})",
     (REG.get("locked") is True) == bool(sign_ok) and (not REG.get("locked") or _filled(REG.get("locked_at")))),
    (f"Version = config.registry.version ({VERSION}); config and funnel hashes recorded",
     str(REG["version"]) == VERSION and REG["config_sha256"] == CONFIG_HASH and REG["funnel_config_hash"] == FUNNEL_HASH),
    ("Decision-rule thresholds equal config.selection",
     sel_reg == canonical(SEL) and rules["R1_discrimination"]["threshold"] == SEL["r1_gini_lift_lower"]
     and rules["R3_time_stability"]["tfold_std_ratio"] == SEL["r3_tfold_std_ratio"]
     and rules["R3_time_stability"]["q2_ks_drop_margin"] == SEL["r3_q2_ks_drop_margin"]
     and str(SEL["c_over_b_gini_lift_lower"]) in REG["pre_registered"]["selection_logic"][3]
     and REG["pre_registered"]["bootstrap"] == canonical(SEL["bootstrap"])),
    ("pre_registered.strategy_overlay equals config.strategy_overlay",
     REG["pre_registered"]["strategy_overlay"] == canonical(OVERLAY)),
    ("Platt calibration fitted on DEV OOF registered; OOF file hashed",
     REG["pre_registered"]["calibration"]["method"] == "platt" and REG["pre_registered"]["calibration"]["fit_on"] == "dev_oof"
     and "oof_predictions.parquet" in REG["files"]),
    (f"Features A ({len(REG['features']['A'])}) and B/C pool ({len(REG['features']['BC_pool'])}) match the feature lists",
     REG["features"]["A"] == A_FEATS and REG["features"]["BC_pool"] == POOL),
    (f"C final rounds {REG['lightgbm_rounds']['final_rounds']}; OOF and T-fold metrics for A, B, C; EBM check recorded",
     REG["lightgbm_rounds"]["final_rounds"] == int(CP["C"]["final_rounds"]) and set(REG["dev_oof_metrics"]) == set("ABC")
     and set(REG["t_fold"]) == set("ABC") and REG["ebm_shape_check"]["result"] in ("PASS", "FAIL")),
    (f"load_window(master, 'OOT') raised before the lock ({PRE_LOCK_TEST['basis']})", PRE_LOCK_TEST["raised"]),
    (f"load_window(master, 'OOT') does not raise after the lock ({POST_LOCK_ROWS:,} OOT rows; labels not loaded)",
     REG.get("locked") is True and POST_LOCK_OK and POST_LOCK_ROWS > 0),
]
PASS_S, FAIL_S = "\u2705 PASS", "\u274c FAIL"
all_pass = True
for desc, ok in checks:
    ok = bool(ok); all_pass &= ok
    print(f"  {PASS_S if ok else FAIL_S} : {desc}")
for n in REG.get("notes", []):
    print(f"  \u26a0 NOTE : {n}")
print(f"  INFO : EBM shape check = {REG['ebm_shape_check']['result']} -> Model B cannot meet R4 as registered"
      if REG["ebm_shape_check"]["result"] == "FAIL" else "")

summary = {"module": "E12_lock", "run_time": datetime.now().isoformat(timespec="seconds"),
           "elapsed_s": round(time.time() - _t0, 1), "git_commit": GIT_COMMIT, "config_hash": CONFIG_HASH,
           "funnel_config_hash": FUNNEL_HASH, "registry_version": VERSION, "action": ACTION,
           "archived_previous": ARCHIVED, "locked": REG.get("locked"), "locked_at": REG.get("locked_at"),
           "signed_by": sig.get("decision_rules_signed_by"), "signed_date": sig.get("signed_date"),
           "n_files_locked": len(REG["files"]), "n_features_A": len(REG["features"]["A"]),
           "n_features_pool": len(REG["features"]["BC_pool"]), "oot_rows_visible_after_lock": POST_LOCK_ROWS,
           "uc_registration": UC_RESULT, "notes": REG.get("notes", []), "mlflow_run_id": MLFLOW_RUN_ID,
           "checks": {d: bool(o) for d, o in checks}, "pass_fail": "PASS" if all_pass else "FAIL"}
write_json(summary, os.path.join(RUN_SUMS, "s04_lock.json"))
print(f"\nRun summary: {os.path.join(RUN_SUMS, 's04_lock.json')}")
for desc, ok in checks:
    assert ok, f"ACCEPTANCE FAILED: {desc}"
print("\n\u2705 All E12 acceptance criteria passed.")